# 01 - date Understanding

## objective

This step aims to understand a structure of the set of date before of the start of the stages analysis exploratória, limpeza and modeling.

The following characteristics will be investigated:

- Dimensionalidade of the dataset;
- types of the variables;
- values missing;
- distributions iniciais;
- statistics descriptive;
- possible inconsistencies;
- definition of the variable target.

At the end of this step will be possible understand a structure geral of the date and identificar the primeiros desafios for a building of the model of forecasting:

- Quantas observations there are?

- Quantas variables there are?

- which are numeric?

- which are categorical?

- there are values missing?

- there are variables redundantes?

- there is alguma variable that not will have to entrar in the model?

- which variable will be utilizada the target?

In [77]:
from pathlib import Path
import pandas as pd

# Caminho of the projeto
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "raw"

# Carregamento of the date
csv_file = DATA_PATH / "data.csv"
df = pd.read_csv(csv_file)

In [78]:
# dimension of the dataset
n_samples, n_features = df.shape

print("=" * 35)
print("dimension of the DATASET")
print("=" * 35)
print(f"Número de observações : {n_samples}")
print(f"Número de atributos   : {n_features}")

DIMENSÃO DO DATASET
Número de observações : 20805
Número de atributos   : 9


In [79]:
df.head()

,produto,descricao,valor_unitario,quantidade,valor_total,data_venda,preco_custo,saldo,grupo
0,25,ANT. UHF GRANDE,"50,00",1.0,"50,00",2/14/2017,"20,00","30,00",ANTENA
1,23,CABO RG06,NaN,15.0,NaN,2/14/2017,NaN,NaN,CABO
2,40,CONECTOR ANILHA RG06,"1,00",1.0,"1,00",2/14/2017,NaN,NaN,CABO
3,38,CONECTOR PRESSAO RG06,NaN,1.0,NaN,2/14/2017,NaN,NaN,CABO
4,65,CONVERSOR DIGITAL BEDIN,"110,00",1.0,"110,00",2/14/2017,"85,00","25,00",CONVERSOR


## structure of the set of date

### values missing and types of variables

In [80]:
print("information sobre the dataset:")
df.info()

Informações sobre o dataset:
<class 'pandas.DataFrame'>
RangeIndex: 20805 entries, 0 to 20804
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   produto         20805 non-null  int64  
 1   descricao       20805 non-null  str    
 2   valor_unitario  18923 non-null  str    
 3   quantidade      20425 non-null  float64
 4   valor_total     19473 non-null  str    
 5   data_venda      20799 non-null  str    
 6   preco_custo     8676 non-null   str    
 7   saldo           9009 non-null   str    
 8   grupo           20805 non-null  str    
dtypes: float64(1), int64(1), str(7)
memory usage: 1.4 MB


The analysis of the structure of the dataset of date initial indica that the set is composto by 20.805 observations distribuídas in nove features. Entre the features disponíveis, we have the seguintes information:

| variable     | type (Pandas) | Categoria         | description                                                          | unit    |
| ------------ | ------------- | ----------------- | ------------------------------------------------------------------ | ---------- |
| **product**  | `int64`         | numeric discreta        | ID of the product.                                                | quantity          |
| **descricao** | `str`         | categorical        | description of the product.                              | —          |
| **value_unitario**     | `float64`       | numeric | value by unit.                                                    | R$         |
| **quantity** | `int64`       | numeric discreta | quantity sold.                                   | quantity |
| **value_total**   | `float64`       | numeric | value total of the sale.                            | R$ |
| **date_sale**     | `datetime`         | date        | date of the sale.                  | date          |
| **preco_cost**     | `float64`       | numeric | value of cost of the product.                                           | R$         |
| **balance**    | `float64`       | numeric | profit sob a sale. | R$         |
| **group**    | `str`       | categorical | group of sale. | —         |

we note that the variables `value_unitario`, `value_total`, `date_sale`, `preco_cost` and `balance` are with the type errado of dado, where the Pandas interpretou the `str` when in the verdade deveria be consideradas variables numeric.

also we have information of that:
$$value\, total = value\,unitario * quantity$$
and
$$balance = value\, total - preco\, cost$$

Adicionalmente, there are many values missing that precisarão of tratamento. there is a possibilidade of completar `value_total`,  `quantity` and `value_unitario` by exister a relação entre elas. however, must-if tomar cuidado with a presence of `quantities` fractional for products that só aceitam inteiros. also disso, not is possible controlar possible **descontos** by sale, because this traria a peso higher for the model of forecasting. 

the set of date also apresenta values of dates missing, must be removed, since iremos tratar of séries temporais. also has-if more of the metadde of the date of `preco_cost` null (~58.30%) and of `balance` null (~56.70%), the that shows that these features not will serve the *features* for the model.

In [81]:
# values missing
print("values missing by coluna:")
print(df.isna().sum())

Valores ausentes por coluna:
produto               0
descricao             0
valor_unitario     1882
quantidade          380
valor_total        1332
data_venda            6
preco_custo       12129
saldo             11796
grupo                 0
dtype: int64


In [82]:
print("number of rows duplicadas:")
print(df.duplicated().sum())

Número de linhas duplicadas:
1816


In [83]:
df[df.duplicated()]

,produto,descricao,valor_unitario,quantidade,valor_total,data_venda,preco_custo,saldo,grupo
5,25,ANT. UHF GRANDE,"50,00",1.0,"50,00",2/14/2017,"20,00","30,00",ANTENA
6,65,CONVERSOR DIGITAL BEDIN,"110,00",1.0,"110,00",2/14/2017,"85,00","25,00",CONVERSOR
11,25,ANT. UHF GRANDE,"50,00",1.0,"50,00",2/14/2017,"20,00","30,00",ANTENA
15,12,CONVERSOR DIGITAL VISIONTEC,"110,00",1.0,"110,00",2/15/2017,"75,00","35,00",CONVERSOR
17,25,ANT. UHF GRANDE,"50,00",1.0,"50,00",2/15/2017,"20,00","30,00",ANTENA
...,...,...,...,...,...,...,...,...,...
20722,278,ANTENA UHF 34 ELEMENTOS,"70,00",1.0,"70,00",8/4/2025,"30,00","40,00",ANTENA
20764,278,ANTENA UHF 34 ELEMENTOS,"70,00",1.0,"70,00",8/13/2025,"30,00","40,00",ANTENA
20767,188,CABO COAXIAL RG 59,"2,50",10.0,"22,00",8/13/2025,NaN,"9,00",CABO
20773,43,CONECTOR ROSCA,"1,00",2.0,"2,00",8/13/2025,NaN,NaN,CABO


Esses values not are duplicatas reais, because are sales existentes and with different IDs of pedidos.

## hypotheses Iniciais

with base in the understanding of the negócio in conjunction with the set of date, a compreensão initial of the variables helps a levantar some hypotheses before of the execution of the Pré-processamento and of the analysis Exploratória of date (EDA). Essas hypotheses will serve the a starting point for guide the próximas stages of the projeto and will have to be confirmadas or refutadas through analyses statistics and visualizações.

### hypothesis 1 - ID of the product

a variable `product` only identifica the product sold. of first momento, considering that is a variable with high cardinalidade, not if shows relevante in the sentido of calcular the revenue total monthly **by product**, because this atrapalharia the model. however, if houver tempo hábil, would be interesting investigar the products with higher faturamentos monthly separadamente and, for this, it is necessary use the identificador of the product.

---

### hypothesis 2 - description of the product

this variable só servirá for understand the negócio the a todo and for analyses específicas. Ela will be interesting for understand possible *outliers* that possam aparecer in the step of EDA.

---

### hypothesis 3 - quantity

not is permitido ter values fractional for esta variable, exceto if houver the understanding of the negócio for products específicos.

---

### hypothesis 4 - group

this variable will be interesting for understand which groups contribuem more for the revenue global of the company mensalmente and understand if there are groups with high cardinalidade that can be removidor without prejudicar a analysis.

---

## Pré-processamento of the date

### step for tratamento of nome and type of variables.

In [84]:
banco_inicial = df.copy()

banco_inicial.rename(columns={'produto': 'id_produto'}, inplace=True)

In [85]:
# Modificando the columns of values monetários for the formato numeric

colunas_monetarias = [
    'valor_unitario',
    'valor_total',
    'preco_custo',
    'saldo'
]

for col in colunas_monetarias:
    banco_inicial[col] = (
        banco_inicial[col]
        .astype(str)
        .str.replace('.', '', regex=False)
        .str.replace(',', '.', regex=False)
    )

    banco_inicial[col] = pd.to_numeric(
            banco_inicial[col],
            errors='coerce' # Converte values not numeric for NaN
        )

In [86]:
# Modificando a coluna of date for the formato datetime
banco_inicial['data_venda'] = pd.to_datetime(
    banco_inicial['data_venda'],
    errors='coerce' # Converte values not convertíveis for NaT
)

In [87]:
banco_inicial.info()

<class 'pandas.DataFrame'>
RangeIndex: 20805 entries, 0 to 20804
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   id_produto      20805 non-null  int64         
 1   descricao       20805 non-null  str           
 2   valor_unitario  18923 non-null  float64       
 3   quantidade      20425 non-null  float64       
 4   valor_total     19473 non-null  float64       
 5   data_venda      20799 non-null  datetime64[us]
 6   preco_custo     8676 non-null   float64       
 7   saldo           9009 non-null   float64       
 8   grupo           20805 non-null  str           
dtypes: datetime64[us](1), float64(5), int64(1), str(2)
memory usage: 1.4 MB


### Tratamento of values null and missing

In [88]:
banco_inicial[
    ['valor_unitario',
     'quantidade',
     'valor_total',
     'preco_custo',
     'saldo']
].isna().sum()

valor_unitario     1882
quantidade          380
valor_total        1332
preco_custo       12129
saldo             11796
dtype: int64

#### analysis of values missing entre `balance` and `preco_cost`

In [89]:
banco_inicial.groupby('grupo')[['preco_custo', 'saldo']].apply(
    lambda x: x.isna().sum()
)

,preco_custo,saldo
grupo,,
ANTENA,1420,1445
CABO,4773,4260
COMPONENTES ANTENAS,2119,2160
CONVERSOR,265,267
INTERFONE,321,323
PORTAO,1081,1074
PRODUTO ESPECIAL,3,1
RECEPTOR,13,16
REDES,2,2


* we note that, many values missing in `preco_cost` and `balance` impossibilitam a analysis of profits of maneira eficiente, therefore, here is a ótima justificativa for discard a ideia of **forecasting of profit** and focar only in *analyses of seasonality*, *forecasts of demand* and *revenue monthly*

#### analyzing values missing of `value_total` and `value_unitario`

* the a variable *value_total* is dada by *value_total = value_unitario * quantity*, is possible fill some values empty, of the mesma forma aplicarmos that *value_unitario = value_total / quantity*

In [90]:
# filling values missing of value_total and value_unitario
mask_total = (
    banco_inicial['valor_total'].isna()
    & banco_inicial['valor_unitario'].notna()
    & banco_inicial['quantidade'].notna()
)

banco_inicial.loc[mask_total, 'valor_total'] = (
    banco_inicial.loc[mask_total, 'valor_unitario']
    * banco_inicial.loc[mask_total, 'quantidade']
)

mask_unitario = (
    banco_inicial['valor_unitario'].isna()
    & banco_inicial['valor_total'].notna()
    & banco_inicial['quantidade'].notna()
    & (banco_inicial['quantidade'] > 0)
)

banco_inicial.loc[mask_unitario, 'valor_unitario'] = (
    banco_inicial.loc[mask_unitario, 'valor_total']
    / banco_inicial.loc[mask_unitario, 'quantidade']
)

In [91]:
# Verificando novamente the values missing após the preenchimento
banco_inicial[
    ['valor_unitario',
     'quantidade',
     'valor_total',
     'preco_custo',
     'saldo']
].isna().sum()

valor_unitario      794
quantidade          380
valor_total         906
preco_custo       12129
saldo             11796
dtype: int64

* Reduziu-if the number of value missing of value_unitario of 1882 for 794 (improvement of 57.81%) and the number of values missing of value_total of 1332 for 906 (improvement of 31.98%).

* by end, retira-if the records with `value_total`, `value_unitario` and `quantity` null or missing.

In [92]:
banco_inicial = banco_inicial.dropna(
    subset=['valor_unitario', 'quantidade', 'valor_total'],
    how='all'
)

In [93]:
# number of records restantes in the dataset of date
print(f"O número de registros restantes no banco de dados é: {banco_inicial.shape[0]}")

O número de registros restantes no banco de dados é: 20802


#### records with `value_total` filled, mas `value_unitario` and `quantity` missing

In [94]:
mask = (
    banco_inicial['valor_total'].notna()
    & banco_inicial['valor_unitario'].isna()
    & banco_inicial['quantidade'].isna()
)

mask.sum()

np.int64(9)

* without the variables `quantity` and `value_unitario` is difficult of perform boas analyses, then it is useful exclude these records, considering that are few and not afetarão the model end.

In [95]:
banco_inicial = banco_inicial[~mask].copy()

#### records with `quantity` vazia, mas `value_unitario` and `value_total` preenchidos

In [96]:
mask_qtdna = (
    banco_inicial['valor_unitario'].notna()
    & banco_inicial['quantidade'].isna()
    & banco_inicial['valor_total'].notna()
)

mask_qtdna.sum()

np.int64(247)

In [97]:
# Calculando a quantity a partir of the value_total and value_unitario
qtd_calc = (
    banco_inicial.loc[mask_qtdna, 'valor_total']
    / banco_inicial.loc[mask_qtdna, 'valor_unitario']
)
print(qtd_calc)

410      2.5
414      2.5
2463     1.3
3285     1.5
7226     0.5
        ... 
20703    0.5
20715    0.5
20719    3.5
20740    0.5
20768    0.5
Length: 247, dtype: float64


In [98]:
# Verificando quantos values calculados are inteiros
print((qtd_calc % 1 == 0).sum())

2


In [99]:
banco_inicial = banco_inicial[~mask_qtdna].copy()

* a quantity calculated shows values fractional, mas this not is consistent with a quantity sold of products nesta loja. the that shows that necessariamente the value charged in the end of the compra (`value_total`) was diferente of `value_unitario` * `quantity`, the that shows possible descontos or reajustes in the ato of the sale.

* the build a variable of reajuste for few date (about of 1.2%) introduziria noise in the forecasts, is válido remove of the dataset of date.

#### records with `value_unitario` vazia, mas `quantity` and `value_total` preenchidos

In [100]:
mask_vuna = (
    banco_inicial['valor_unitario'].isna()
    & banco_inicial['quantidade'].notna()
    & banco_inicial['valor_total'].notna()
)

mask_vuna.sum()

np.int64(0)

#### records with `value_unitario`, `quantity` and `value_total` empty

In [101]:
mask_numna = (
    banco_inicial['valor_unitario'].isna()
    & banco_inicial['quantidade'].isna()
    & banco_inicial['valor_total'].isna()
)

mask_numna.sum()

np.int64(0)

#### analysis of dates

In [102]:
# number of records restantes in the dataset of date
print(f"O número de registros restantes no banco de dados é: {banco_inicial.shape[0]}")

O número de registros restantes no banco de dados é: 20546


In [103]:
banco_inicial.info()

<class 'pandas.DataFrame'>
Index: 20546 entries, 0 to 20804
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   id_produto      20546 non-null  int64         
 1   descricao       20546 non-null  str           
 2   valor_unitario  19764 non-null  float64       
 3   quantidade      20425 non-null  float64       
 4   valor_total     19643 non-null  float64       
 5   data_venda      20540 non-null  datetime64[us]
 6   preco_custo     8636 non-null   float64       
 7   saldo           8995 non-null   float64       
 8   grupo           20546 non-null  str           
dtypes: datetime64[us](1), float64(5), int64(1), str(2)
memory usage: 1.6 MB


there are `value_unitario`, `quantity`, `value_total` and `date_sale` null, the which not will be tratados due tos limitações of the dataset disponível. Logo, must be removed. 

In [105]:
# number of records with date of sale missing
mask_datna = (
    banco_inicial['data_venda'].isna()
)

mask_datna.sum()

np.int64(6)

In [106]:
# Retirando the records with dates missing
banco_inicial = banco_inicial[~mask_datna].copy()

#### quantities null or negative

In [107]:
# number of records with quantity lower or igual a zero
banco_inicial[banco_inicial['quantidade'] <= 0].shape[0]

0

In [108]:
# number of records with values_unitarios lower or igual a zero
banco_inicial[banco_inicial['valor_unitario'] <= 0].shape[0]

3

In [109]:
banco_inicial[banco_inicial['valor_unitario'] <= 0]

,id_produto,descricao,valor_unitario,quantidade,valor_total,data_venda,preco_custo,saldo,grupo
3658,42,EMENDA,0.0,3.0,0.0,2018-04-23,NaN,NaN,CABO
5877,101,SUP CAVALETE 1,0.0,1.0,0.0,2019-03-22,NaN,NaN,COMPONENTES ANTENAS
7871,0,FRENTE AGL 16 PONTOS,0.0,1.0,0.0,2020-07-29,0.0,0.0,PRODUTO ESPECIAL


In [110]:
# number of records with value_total lower or igual a zero
banco_inicial[banco_inicial['valor_total'] <= 0].shape[0]

3

In [111]:
banco_inicial[banco_inicial['valor_total'] <= 0]

,id_produto,descricao,valor_unitario,quantidade,valor_total,data_venda,preco_custo,saldo,grupo
3658,42,EMENDA,0.0,3.0,0.0,2018-04-23,NaN,NaN,CABO
5877,101,SUP CAVALETE 1,0.0,1.0,0.0,2019-03-22,NaN,NaN,COMPONENTES ANTENAS
7871,0,FRENTE AGL 16 PONTOS,0.0,1.0,0.0,2020-07-29,0.0,0.0,PRODUTO ESPECIAL


In [112]:
# Retirando the inconsistencies of quantity, value_unitario and value_total
banco_inicial.drop([3658, 5877, 7871], inplace=True)

In [113]:
banco_inicial.reset_index(drop=True, inplace=True)

In [114]:
banco_inicial.info()

<class 'pandas.DataFrame'>
RangeIndex: 20537 entries, 0 to 20536
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   id_produto      20537 non-null  int64         
 1   descricao       20537 non-null  str           
 2   valor_unitario  19755 non-null  float64       
 3   quantidade      20416 non-null  float64       
 4   valor_total     19634 non-null  float64       
 5   data_venda      20537 non-null  datetime64[us]
 6   preco_custo     8631 non-null   float64       
 7   saldo           8990 non-null   float64       
 8   grupo           20537 non-null  str           
dtypes: datetime64[us](1), float64(5), int64(1), str(2)
memory usage: 1.4 MB


In [115]:
# Retirando the records with values missing in quantity, value_unitario and value_total
banco_inicial.dropna(subset=['valor_unitario', 'quantidade', 'valor_total'], inplace=True)

In [116]:
banco_inicial.info()

<class 'pandas.DataFrame'>
Index: 19634 entries, 0 to 20536
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   id_produto      19634 non-null  int64         
 1   descricao       19634 non-null  str           
 2   valor_unitario  19634 non-null  float64       
 3   quantidade      19634 non-null  float64       
 4   valor_total     19634 non-null  float64       
 5   data_venda      19634 non-null  datetime64[us]
 6   preco_custo     8458 non-null   float64       
 7   saldo           8932 non-null   float64       
 8   grupo           19634 non-null  str           
dtypes: datetime64[us](1), float64(5), int64(1), str(2)
memory usage: 1.5 MB


In [117]:
# number of records restantes in the dataset of date
print(f"O número de registros restantes no banco de dados é: {banco_inicial.shape[0]}")

O número de registros restantes no banco de dados é: 19634


In [126]:
banco_inicial.to_csv(
    "../data/interm/data_1.csv",
    index=False
)

---

## Primeiras analyses statistics

In [118]:
banco_inicial.describe()

,id_produto,valor_unitario,quantidade,valor_total,data_venda,preco_custo,saldo
count,19634.000000,19634.000000,19634.000000,19634.000000,19634,8458.000000,8932.000000
mean,218.267342,72.463404,4.183559,96.755103,2021-04-28 20:04:12.297035,69.029558,46.740596
min,0.000000,0.500000,1.000000,1.000000,2017-02-14 00:00:00,0.000000,-14179.000000
25%,44.000000,5.000000,1.000000,16.000000,2018-12-03 00:00:00,15.000000,13.000000
50%,233.000000,35.000000,1.000000,44.000000,2021-05-31 00:00:00,25.000000,25.000000
75%,318.000000,70.000000,2.000000,90.000000,2023-05-31 00:00:00,70.000000,40.000000
max,737.000000,7536.000000,400.000000,9720.000000,2025-10-30 00:00:00,17160.000000,7560.000000
std,167.363054,172.379914,12.511824,244.465625,NaN,291.181147,242.768560


### Interpretação of the statistics descriptive

the variables `preco_cost` and `balance` apresentam many records with values missing and inconsistencies, the values null and negative. these variables will be descartadas of the model.

a variable `value_unitario` possui a mediana (R$ 35) abaixo of the mean (R$ 72.46), the that shows a distribution positiva, this is, assimétrica a direita. also is possible comprovar this observing the desvio padrão and the value máximo of the variable, the that deixa evidente a presence of possible *outliers*.


of maneira similar, a variable `value_total` also possui a assimetria direita, since ela is construída a partir of the variable `value_unitario`.

---

# Conclusão

Apesar of be a boa base of date for if create models of forecasting, it is necessary some tratamentos. 

1) all the groups must be analisados in the revenue?

2) a high cardinalidade of different products irá afetar a modeling? if sim, compensa pensar in revenue by group and not by products individualmente.

3) the revenue is justamente a variable `value_total`, logo ela is construída pelas variables `quantity` and `value_unitario`, this significa that is ocorrendo **date leakage**?

4) which normalizações must be feitas in the variables numeric for tratamento of values extremos?